# DarkLens NLP Classifier — Google Colab Training

Fine-tunes **DistilBERT** to classify dark-pattern UX text into 10 categories.

### Before running
1. Set runtime: **Runtime → Change runtime type → T4 GPU** (free tier is enough)
2. On your **local machine** run:
   ```bash
   python scripts/prepare_nlp_dataset.py
   ```
   This creates `data/nlp/train.csv` and `data/nlp/val.csv`.
3. Upload both CSV files to **Google Drive** (e.g. `My Drive/darklens/data/nlp/`)
4. Edit the paths in **Cell 3**, then run all cells in order ↓


In [ ]:
# Cell 1 — Check GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU is available!')
    print(r.stdout[:300])
else:
    print('No GPU found. Go to Runtime > Change runtime type > T4 GPU')


In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')


In [ ]:
# Cell 3 — Configure paths  (EDIT THESE to match your Drive folder)
TRAIN_CSV  = '/content/drive/MyDrive/darklens/data/nlp/train.csv'
VAL_CSV    = '/content/drive/MyDrive/darklens/data/nlp/val.csv'
OUTPUT_DIR = '/content/drive/MyDrive/darklens/models/nlp_classifier'

# Training hyperparameters — defaults are good for T4 GPU
BASE_MODEL    = 'distilbert-base-uncased'
EPOCHS        = 5
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
MAX_SEQ_LEN   = 64
SEED          = 42

import os
assert os.path.exists(TRAIN_CSV), f'train.csv not found: {TRAIN_CSV}'
assert os.path.exists(VAL_CSV),   f'val.csv not found: {VAL_CSV}'
print(f'Files found. Model will be saved to: {OUTPUT_DIR}')


In [ ]:
# Cell 4 — Install dependencies
!pip install -q 'transformers>=4.45' 'datasets>=3.0' 'scikit-learn>=1.5' 'accelerate>=0.26'
print('Dependencies installed')


In [ ]:
# Cell 5 — Preview data and class distribution
import pandas as pd
from collections import Counter

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
print(f'Train: {len(train_df)} rows | Val: {len(val_df)} rows')

all_labels = sorted(set(train_df['label']) | set(val_df['label']))
tc, vc = Counter(train_df['label']), Counter(val_df['label'])
print(f"\n{'Label':<35} {'Train':>6} {'Val':>6} {'Total':>7}")
print('-' * 58)
for lbl in all_labels:
    t, v = tc[lbl], vc[lbl]
    print(f'{lbl:<35} {t:>6} {v:>6} {t+v:>7}')

print('\nSample rows:')
display(train_df.sample(5, random_state=42)[['text', 'label']])


In [ ]:
# Cell 6 — Label mappings (must match domain/nlp/labels.py NlpLabel exactly)
ALL_LABELS = [
    'normal', 'confirmshaming', 'false_urgency', 'scarcity',
    'fear_based_copy', 'hidden_subscription', 'misleading_consent',
    'emotional_manipulation', 'false_discount', 'forced_continuity',
]

present = sorted(set(train_df['label']) | set(val_df['label']))
missing = set(ALL_LABELS) - set(present)
if missing:
    print(f'WARNING: No training data for: {missing}')
    print('  The model cannot detect these. Collect more data to fix this.')
    ALL_LABELS = [l for l in ALL_LABELS if l in present]

label2id = {l: i for i, l in enumerate(ALL_LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(f'Using {len(ALL_LABELS)} classes: {ALL_LABELS}')


In [ ]:
# Cell 7 — Tokenize dataset
import torch
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def df_to_dataset(df):
    df = df[df['label'].isin(label2id)].copy()
    df['label'] = df['label'].map(label2id)
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_SEQ_LEN)

train_ds = df_to_dataset(train_df).map(tokenize_fn, batched=True)
val_ds   = df_to_dataset(val_df).map(tokenize_fn, batched=True)
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

device_str = 'GPU' if torch.cuda.is_available() else 'CPU (WARNING: very slow, switch to GPU)'
print(f'Train={len(train_ds)} | Val={len(val_ds)} | Device={device_str}')


In [ ]:
# Cell 8 — Fine-tune DistilBERT  (takes ~10-15 min on T4 GPU)
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = float((preds == labels).mean())
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1_macro': f1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(ALL_LABELS),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=SEED,
    report_to=[],
    fp16=torch.cuda.is_available(),
    logging_steps=50,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting training ...')
trainer.train()
print('Training complete!')


In [ ]:
# Cell 9 — Per-class evaluation + save metrics
import json
from sklearn.metrics import classification_report, confusion_matrix

pred_out    = trainer.predict(val_ds)
preds       = np.argmax(pred_out.predictions, axis=-1)
true_labels = pred_out.label_ids
label_names = [id2label[i] for i in range(len(ALL_LABELS))]

print('=' * 65)
print('PER-CLASS REPORT  (copy these into docs/nlp_experiments.md)')
print('=' * 65)
print(classification_report(true_labels, preds, target_names=label_names, zero_division=0))

os.makedirs(OUTPUT_DIR, exist_ok=True)
report = classification_report(true_labels, preds, target_names=label_names,
                                output_dict=True, zero_division=0)
with open(f'{OUTPUT_DIR}/eval_metrics.json', 'w') as fh:
    json.dump(report, fh, indent=2)

cm = confusion_matrix(true_labels, preds).tolist()
with open(f'{OUTPUT_DIR}/confusion_matrix.json', 'w') as fh:
    json.dump({'labels': label_names, 'matrix': cm}, fh, indent=2)

print(f'Metrics saved to {OUTPUT_DIR}')


In [ ]:
# Cell 10 — Save model to Google Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('Model saved to Google Drive:', OUTPUT_DIR)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    tag = f'{sz/1024/1024:.1f} MB' if sz > 1048576 else f'{sz/1024:.1f} KB'
    print(f'  {fname:<45} {tag}')


In [ ]:
# Cell 11 — Download model as ZIP to your laptop
# Only run if you want a local copy in addition to the Drive copy.
import shutil
from google.colab import files

shutil.make_archive('/content/nlp_classifier_model', 'zip', OUTPUT_DIR)
print('Zipped. Downloading...')
files.download('/content/nlp_classifier_model.zip')
print('Done!')
print('Extract to: d:\\Projects\\darklens\\models\\nlp_classifier\\')
print('DarkLens auto-detects the model on the next scan run.')


## After training is done

1. **Copy model to your local project**
   Extract the downloaded zip so the path is:
   `d:\Projects\darklens\models\nlp_classifier\`

2. **Run a scan** — the NLP WARNING will be gone:
   ```bash
   python -m darklens.interfaces.cli.main https://example.com --formats json,html
   ```

3. **Update `docs/nlp_experiments.md`**
   Copy F1/precision/recall numbers from Cell 9 output.

4. **Tune confidence threshold**
   Update `nlp_confidence_threshold` in `config.py` based on your precision-recall curve.
